<a href="https://colab.research.google.com/github/DiliniSew/SinVL_FYP/blob/dataset/Dataset_Creation(V5)_FYP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("deep-translator")
install("kagglehub")
install("pandas")

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import time
import random
import shutil
import pandas as pd
from collections import defaultdict
from deep_translator import GoogleTranslator
import kagglehub

In [ ]:
BATCH_SIZE        = 100
DEST_LANG         = 'si'     # Sinhala


DRIVE_DIR         = '/content/drive/MyDrive/SinVL'
WORKING_COPY      = os.path.join(DRIVE_DIR, 'captions_working_copy.txt')
OUTPUT_DIR        = os.path.join(DRIVE_DIR, 'batches')
ORIGINAL_FILENAME = 'captions.txt'
DELAY_SECONDS     = 0.3
MAX_RETRIES       = 3


os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[Drive]  Working folder → {DRIVE_DIR}")

[Drive]  Working folder → /content/drive/MyDrive/SinVL


In [ ]:
def setup_working_copy():
    print("\n[Setup]  Loading Flickr30k from Kaggle...")
    dataset_path = kagglehub.dataset_download("adityajn105/flickr30k")
    original     = os.path.join(dataset_path, ORIGINAL_FILENAME)

    if not os.path.exists(WORKING_COPY):
        print("[Setup]  Creating working copy on Drive (one-time, may take a moment)...")
        shutil.copy2(original, WORKING_COPY)
        print(f"[Setup]  ✅ Working copy created → '{WORKING_COPY}'")
    else:
        print(f"[Setup]  ✅ Working copy found on Drive → '{WORKING_COPY}'")

    return original

In [ ]:
# Load captions
def load_captions_from_file(filepath):
    image_captions = defaultdict(list)
    all_image_ids  = []
    raw_rows       = []

    with open(filepath, 'r', encoding='utf-8') as f:
        header = f.readline()
        for line in f:
            parts = line.strip().split(',', 1)
            if len(parts) != 2:
                continue
            img_id_raw, caption = parts
            img_id = img_id_raw.split('#')[0]
            image_captions[img_id].append(caption.strip())
            if img_id not in all_image_ids:
                all_image_ids.append(img_id)
            raw_rows.append((img_id_raw, caption.strip()))

    return image_captions, all_image_ids, raw_rows, header

In [ ]:
# Delete translated images
def remove_from_working_copy(translated_ids, raw_rows, header):
    kept = [
        (img_id_raw, caption)
        for (img_id_raw, caption) in raw_rows
        if img_id_raw.split('#')[0] not in translated_ids
    ]
    with open(WORKING_COPY, 'w', encoding='utf-8') as f:
        f.write(header)
        for img_id_raw, caption in kept:
            f.write(f"{img_id_raw},{caption}\n")
    return len(kept)

In [ ]:
# Progress
def show_progress(total_images, remaining, batch_number):
    translated_so_far = total_images - remaining
    pct_done  = translated_so_far / total_images * 100
    bar_width = 42
    filled    = int(bar_width * translated_so_far / total_images)
    bar       = "█" * filled + "░" * (bar_width - filled)

    print(f"""
┌──────────────────────────────────────────────────────┐
│          SinVL  —  Translation Progress              │
├──────────────────────────────────────────────────────┤
│  Total images    : {total_images:>7,}                        │
│  ✅ Translated    : {translated_so_far:>7,}   ({pct_done:5.1f}%)             │
│  🕒 Remaining     : {remaining:>7,}                        │
│  📦 Batches done : {batch_number - 1:>7}                        │
│  📦 Batches left : ~{-(-remaining // BATCH_SIZE):<7}                       │
├──────────────────────────────────────────────────────┤
│  [{bar}]  │
└──────────────────────────────────────────────────────┘""")

In [ ]:
# Translate a single caption with retry
def translate_caption(text, translator):
    for attempt in range(MAX_RETRIES):
        try:
            result = translator.translate(text)
            return result if result else ""
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep(1.5 * (attempt + 1))
            else:
                print(f"  ⚠️  Failed: '{text[:55]}' — {e}")
                return ""
    return ""


In [ ]:
# Translate selected batch
def translate_batch(selected_ids, image_captions):
    translator = GoogleTranslator(source='en', target=DEST_LANG)
    rows   = []
    total  = sum(len(image_captions[img]) for img in selected_ids)
    count  = 0
    failed = 0

    print(f"\n[Translate] {total} captions ({len(selected_ids)} images × ~5 each)...")
    print("[Translate] Please wait...\n")

    for img_id in selected_ids:
        for idx, caption in enumerate(image_captions[img_id]):
            sinhala = translate_caption(caption, translator)
            if not sinhala:
                failed += 1

            rows.append({
                'image_id':         img_id,
                'caption_index':    idx,
                'original_caption': caption,
                'sinhala_caption':  sinhala,
            })

            count += 1
            if count % 100 == 0:
                print(f"  ↳ {count}/{total} captions done  (failed: {failed})")

            time.sleep(DELAY_SECONDS)

    print(f"\n[Translate] ✅ {count} captions done  |  ⚠️  {failed} failed")
    return rows, failed

In [ ]:
# Save batch CSV
def save_batch_csv(rows, batch_number):
    filename = os.path.join(OUTPUT_DIR, f"translated_batch_{batch_number:03d}.csv")
    df = pd.DataFrame(rows, columns=[
        'image_id', 'caption_index', 'original_caption', 'sinhala_caption'
    ])
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"[Save]   ✅ CSV saved → '{filename}'  ({len(df)} rows)")
    return filename


# Get next batch number
def get_next_batch_number():
    existing = [
        f for f in os.listdir(OUTPUT_DIR)
        if f.startswith('translated_batch_') and f.endswith('.csv')
    ]
    return len(existing) + 1




In [ ]:
# Main
def main():
    print("=" * 55)
    print("  SinVL — Flickr30k Sinhala Translation Pipeline v4")
    print("=" * 55)

    # Setup working copy on Drive
    original_path = setup_working_copy()

    # Total images count
    _, all_original_ids, _, _ = load_captions_from_file(original_path)
    total_images = len(all_original_ids)

    # Remaining images
    image_captions, remaining_ids, raw_rows, header = load_captions_from_file(WORKING_COPY)
    remaining_count = len(remaining_ids)

    batch_number = get_next_batch_number()

    print(f"\n[Info]  Total images in dataset  : {total_images:,}")
    print(f"[Info]  Remaining (not yet done) : {remaining_count:,}")
    print(f"[Info]  This will be batch #{batch_number}")

    # Show progress before this batch
    show_progress(total_images, remaining_count, batch_number)

    if not remaining_ids:
        print("\n✅ All images have been translated! Nothing left to do.")
        return

    # Select next batch
    n = min(BATCH_SIZE, remaining_count)
    if n < BATCH_SIZE:
        print(f"\n⚠️  Only {n} images left — translating all remaining.")
    selected_ids = random.sample(remaining_ids, n)
    print(f"\n[Batch]  Selected {len(selected_ids)} images for batch #{batch_number}.")

    # Translate
    rows, failed_count = translate_batch(selected_ids, image_captions)

    # Save CSV to Drive
    output_file = save_batch_csv(rows, batch_number)

    # Remove translated rows from working copy on Drive
    kept_rows = remove_from_working_copy(set(selected_ids), raw_rows, header)
    new_remaining = kept_rows // 5

    print(f"[Copy]   Removed {len(selected_ids)} images from working copy.")

    # Show updated progress after this batch
    show_progress(total_images, new_remaining, batch_number + 1)

    print(f"""
  ✅  Batch #{batch_number} complete!
  📄  Saved to  : {output_file}
  ⚠️   Failed captions : {failed_count} / {len(rows)}

  ▶  Run this cell again for the next batch.
""")



main()


  SinVL — Flickr30k Sinhala Translation Pipeline v4

[Setup]  Loading Flickr30k from Kaggle...
Using Colab cache for faster access to the 'flickr30k' dataset.
[Setup]  ✅ Working copy found on Drive → '/content/drive/MyDrive/SinVL/captions_working_copy.txt'

[Info]  Total images in dataset  : 31,783
[Info]  Remaining (not yet done) : 20,283
[Info]  This will be batch #3

┌──────────────────────────────────────────────────────┐
│          SinVL  —  Translation Progress              │
├──────────────────────────────────────────────────────┤
│  Total images    :  31,783                        │
│  ✅ Translated    :  11,500   ( 36.2%)             │
│  🕒 Remaining     :  20,283                        │
│  📦 Batches done :       2                        │
│  📦 Batches left : ~203                           │
├──────────────────────────────────────────────────────┤
│  [███████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░]  │
└──────────────────────────────────────────────────────┘

[Batch]  Selected 100 